# 🔗 Building RAG and Planning Harness

This notebook builds and tests a RAG (Retrieval-Augmented Generation) system
and a LangGraph-based planning harness for τ-Knowledge banking_knowledge.

## Architecture

```
User request → Planning → KB retrieval/tool discovery → Policy application →
Tool call → Result verification → Response/follow-up question
```

## Components

| Component | Role |
|----------|------|
| **KB Index** | Vector search over policy, product, and procedure documents |
| **Example Index** | Reference reasoning patterns from training data (optional) |
| **BM25** | Keyword-based retrieval (diagnostic baseline) |
| **LangGraph** | State-based planning and execution graph |
| **FastAPI** | `/v1/agent/step`, `/v1/qa`, `/healthz` endpoints |
| **τ Adapter** | Official simulation tool integration |

### Two Modes
- **`simple_rag`**: Fixed retrieval per turn → response (no additional planning/repair)
- **`agent_rag`**: Explicit planning + additional retrieval + verification loop

In [ ]:
"""Build KB index from bundle."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_rag_config, load_bundle_config, PROJECT_ROOT,
)
from rhoai_model_training_lab.data import BundleManager

load_env()

rag_config = load_rag_config()
release_config = load_bundle_config()
bundle_base = release_config.get("bundle", {}).get("base_path", "data/prepared/tau-knowledge-v1")
bundle_path = PROJECT_ROOT / bundle_base

mgr = BundleManager.load_bundle(bundle_path)
kb_docs = mgr.get_kb_documents()
print(f"KB document count: {len(kb_docs)}")

# Build vector index
embedding_model_id = rag_config["retrieval"]["embedding"]["model_id"]
persist_dir = PROJECT_ROOT / rag_config["retrieval"]["vector_store"]["persist_directory"]
chunk_size = rag_config["retrieval"]["kb_index"]["chunk_size"]
chunk_overlap = rag_config["retrieval"]["kb_index"]["chunk_overlap"]

print(f"\nEmbedding model: {embedding_model_id}")
print(f"Vector store: {persist_dir}")
print(f"Chunk size: {chunk_size}, overlap: {chunk_overlap}")

# Initialize embedding model
from sentence_transformers import SentenceTransformer

print("\nLoading embedding model...")
embed_model = SentenceTransformer(embedding_model_id)
embedding_dim = embed_model.get_sentence_embedding_dimension()
print(f"  Embedding dimension: {embedding_dim}")

# Initialize ChromaDB
import chromadb

persist_dir.mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(persist_dir))

# Create/reset KB collection
kb_collection_name = rag_config["retrieval"]["kb_index"]["collection_name"]
try:
    chroma_client.delete_collection(kb_collection_name)
except Exception:
    pass

kb_collection = chroma_client.create_collection(
    name=kb_collection_name,
    metadata={"hnsw:space": "cosine"},
)

# Chunk and index documents
print("\nChunking and indexing documents...")
total_chunks = 0
for doc in kb_docs:
    text = doc.get("text", "")
    doc_id = doc.get("document_id", doc.get("id", ""))

    # Simple chunking with overlap
    chunks = []
    for start in range(0, len(text), chunk_size - chunk_overlap):
        chunk_text = text[start:start + chunk_size]
        if chunk_text.strip():
            chunks.append(chunk_text)

    if not chunks:
        continue

    embeddings = embed_model.encode(chunks, show_progress_bar=False).tolist()

    ids = [f"{doc_id}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [{"document_id": doc_id, "chunk_index": i} for i in range(len(chunks))]

    kb_collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=chunks,
        metadatas=metadatas,
    )
    total_chunks += len(chunks)

print(f"\n✅ KB index build complete")
print(f"   Total documents: {len(kb_docs)}")
print(f"   Total chunks: {total_chunks}")
print(f"   Storage location: {persist_dir}")

In [ ]:
"""Test retrieval — dense search and BM25 comparison."""

from rank_bm25 import BM25Okapi

# Test queries
test_queries = [
    "What is the maximum daily transfer limit?",
    "How do I dispute a transaction?",
    "What are the requirements for opening a premium account?",
]

print("=" * 70)
print("🔍 Retrieval Test: Dense (vector) vs BM25 (keyword) Comparison")
print("=" * 70)

# Build BM25 index for comparison
all_chunks = kb_collection.get(include=["documents"])
chunk_texts = all_chunks["documents"]
chunk_ids = all_chunks["ids"]

tokenized_corpus = [doc.lower().split() for doc in chunk_texts]
bm25 = BM25Okapi(tokenized_corpus)

for query in test_queries:
    print(f"\n📝 Query: \"{query}\"")

    # Dense retrieval
    query_embedding = embed_model.encode([query]).tolist()
    dense_results = kb_collection.query(
        query_embeddings=query_embedding,
        n_results=3,
    )

    print("\n  🔷 Dense retrieval results (top 3):")
    for i, (doc, dist) in enumerate(
        zip(dense_results["documents"][0], dense_results["distances"][0])
    ):
        score = 1 - dist  # cosine similarity
        print(f"    {i+1}. [Similarity: {score:.3f}] {doc[:120]}...")

    # BM25 retrieval
    bm25_scores = bm25.get_scores(query.lower().split())
    top_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:3]

    print("\n  🔶 BM25 retrieval results (top 3):")
    for rank, idx in enumerate(top_indices):
        print(f"    {rank+1}. [Score: {bm25_scores[idx]:.3f}] {chunk_texts[idx][:120]}...")

    print(f"  {'─' * 60}")

In [ ]:
"""Start backend and test /healthz."""

import subprocess
import sys
import time
import httpx

backend_host = os.environ.get("BACKEND_HOST", "127.0.0.1")
backend_port = int(os.environ.get("BACKEND_PORT", "8000"))
backend_url = f"http://{backend_host}:{backend_port}"

print("=" * 70)
print("🚀 Backend Startup and Health Check")
print("=" * 70)

# Check if already running
try:
    resp = httpx.get(f"{backend_url}/healthz", timeout=5)
    if resp.status_code == 200:
        print(f"✅ Backend is already running: {backend_url}")
        health = resp.json()
        print(f"   Status: {health}")
        backend_running = True
    else:
        backend_running = False
except Exception:
    backend_running = False

if not backend_running:
    # Try starting via script
    start_script = PROJECT_ROOT / "scripts" / "start_backend.sh"
    if start_script.exists():
        print(f"\nStarting backend... (port: {backend_port})")
        proc = subprocess.Popen(
            ["bash", str(start_script), "--host", backend_host, "--port", str(backend_port)],
            cwd=str(PROJECT_ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )

        # Wait for startup
        for i in range(30):
            time.sleep(1)
            try:
                resp = httpx.get(f"{backend_url}/healthz", timeout=3)
                if resp.status_code == 200:
                    print(f"\n✅ Backend started: {backend_url}")
                    backend_running = True
                    break
            except Exception:
                print(".", end="", flush=True)
    else:
        print("\n⚠️  Start script not found.")
        print("   Manual start: bash scripts/start_backend.sh")
        print("   Or start directly with uvicorn:")
        print(f"   uvicorn rhoai_model_training_lab.api:app --host {backend_host} --port {backend_port}")

if not backend_running:
    print("\n⚠️  Backend did not start.")
    print("   Subsequent cells can be skipped without a running backend.")

In [ ]:
"""Test /v1/qa endpoint."""

print("=" * 70)
print("🧪 /v1/qa Endpoint Test")
print("=" * 70)

qa_tests = [
    {
        "question": "What is the daily transfer limit for standard accounts?",
        "mode": "rag",
    },
    {
        "question": "How do I report a stolen card?",
        "mode": "rag",
    },
]

if backend_running:
    for i, test in enumerate(qa_tests, 1):
        print(f"\n--- Test {i}: {test['question'][:50]}... ---")
        try:
            resp = httpx.post(
                f"{backend_url}/v1/qa",
                json=test,
                timeout=30,
            )
            resp.raise_for_status()
            result = resp.json()

            print(f"  Status: {result.get('status', 'N/A')}")
            print(f"  Response: {result.get('answer', 'N/A')[:200]}")
            if result.get("citations"):
                print(f"  Citations: {len(result['citations'])}")
                for cit in result["citations"][:2]:
                    print(f"    - {cit.get('document_id', 'N/A')}: {cit.get('text_excerpt', '')[:80]}")
            if result.get("usage"):
                usage = result["usage"]
                print(f"  Usage: {usage.get('total_tokens', 0)} tokens")
                print(f"  Retrieval latency: {usage.get('retrieval_latency_ms', 0):.0f}ms")
        except Exception as exc:
            print(f"  ❌ Failed: {exc}")
else:
    print("⏭️ Backend is not running, skipped.")

In [ ]:
"""Test /v1/agent/step with a sample scenario."""

print("=" * 70)
print("🧪 /v1/agent/step Agent Step Test")
print("=" * 70)

if backend_running:
    # Sample banking scenario
    agent_request = {
        "session_id": "test-session-001",
        "request_id": "step-001",
        "messages": [
            {"role": "user", "content": "I want to transfer $5000 to account 987654. Is that within my daily limit?"},
        ],
        "available_tools": [
            {
                "type": "function",
                "function": {
                    "name": "get_account_info",
                    "description": "Get account information including balance and limits",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "account_id": {"type": "string"}
                        },
                        "required": ["account_id"],
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "transfer_funds",
                    "description": "Transfer funds between accounts",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "from_account": {"type": "string"},
                            "to_account": {"type": "string"},
                            "amount": {"type": "number"},
                        },
                        "required": ["from_account", "to_account", "amount"],
                    },
                },
            },
        ],
        "mode": "agent_rag",
        "knowledge_access": "rag",
    }

    try:
        resp = httpx.post(
            f"{backend_url}/v1/agent/step",
            json=agent_request,
            timeout=60,
        )
        resp.raise_for_status()
        result = resp.json()

        print(f"\n  Session ID: {result.get('session_id')}")
        print(f"  Status: {result.get('status')}")

        if result.get("plan"):
            print(f"\n  📋 Plan: {result['plan'][:300]}")

        if result.get("message"):
            msg = result["message"]
            print(f"\n  🤖 Response [{msg.get('role', '?')}]: {msg.get('content', '')[:300]}")

        if result.get("tool_calls"):
            print(f"\n  🔧 Tool calls:")
            for tc in result["tool_calls"]:
                fn = tc.get("function", tc)
                print(f"    → {fn.get('name', '?')}({json.dumps(fn.get('arguments', {}), ensure_ascii=False)[:100]})")

        if result.get("citations"):
            print(f"\n  📎 Citations: {len(result['citations'])}")

        print(f"\n  Trace ID: {result.get('trace_id', 'N/A')}")

    except Exception as exc:
        print(f"\n  ❌ Failed: {exc}")
else:
    print("⏭️ Backend is not running, skipped.")

In [ ]:
"""Compare simple_rag vs agent_rag modes."""

import json

print("=" * 70)
print("⚖️ simple_rag vs agent_rag Mode Comparison")
print("=" * 70)

comparison_query = {
    "session_id": "compare-test",
    "messages": [
        {"role": "user", "content": "I need to check if I can get a premium credit card upgrade and what the requirements are."},
    ],
    "knowledge_access": "rag",
}

if backend_running:
    for mode in ["simple_rag", "agent_rag"]:
        print(f"\n{'─' * 60}")
        print(f"📋 Mode: {mode}")
        print(f"{'─' * 60}")

        request = {**comparison_query, "mode": mode, "request_id": f"cmp-{mode}"}

        try:
            resp = httpx.post(
                f"{backend_url}/v1/agent/step",
                json=request,
                timeout=60,
            )
            resp.raise_for_status()
            result = resp.json()

            if result.get("plan"):
                print(f"  Plan: {result['plan'][:200]}")
            else:
                print(f"  Plan: (none — expected for simple_rag)")

            if result.get("message"):
                content = result["message"].get("content", "")
                print(f"  Response: {content[:300]}")

            if result.get("tool_calls"):
                print(f"  Tool calls: {len(result['tool_calls'])}")

            if result.get("usage"):
                usage = result["usage"]
                print(f"  Tokens: {usage.get('total_tokens', 0)}")
                print(f"  Total latency: {usage.get('total_latency_ms', 0):.0f}ms")

        except Exception as exc:
            print(f"  ❌ {exc}")

    print(f"\n{'=' * 70}")
    print("Comparison Summary:")
    print("  • simple_rag: Fixed retrieval per turn, no additional planning → fast but simple")
    print("  • agent_rag: Explicit planning, additional retrieval, verification loop → slower but sophisticated")
    print("  • Performance of both modes will be compared in the official evaluation.")
else:
    print("⏭️ Backend is not running, skipped.")
    print("\n📝 Mode Comparison Description:")
    print("  • simple_rag: Fixed retrieval per turn → response/tool call (no planning)")
    print("  • agent_rag: Planning → retrieval → policy application → tool call → verification")
    print("  • Same model, same retriever, same business tools")
    print("  • Only agent_rag allows explicit planning and additional retrieval")

print("\nNext steps:")
print("  📓 07_evaluate.ipynb — Run evaluation")